# Local LLM: data preparation and guardrail demo (no GPU needed)

This notebook prepares and checks the **training data** and demonstrates the guarded tool router. It does **not** train or evaluate a model. Training, evaluation and `generate()` live in `bist_llm_training.ipynb` (Google Colab, T4 GPU). See `notebooks/README.md`.

- Generates the version-controlled A-E seed records (tool selection, evidence reports, safe refusals) and the train/validation/test split.
- Validates reviewed records (`data/llm/reviewed_records.jsonl`) before any training run.
- Shows how a model's JSON tool request is checked by the router and the guardrail metrics.
- **Academic rule:** do not train on cloud-LLM-generated answers. Every real training example must be reviewed by the group.

## 1. Project location

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # notebook opened from notebooks/
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run from the repository folder that contains src/.')
os.chdir(PROJECT_ROOT)

In [ ]:
import json
import random

SEED = 481
random.seed(SEED)
DATASET_DIR = PROJECT_ROOT / 'data' / 'llm' / 'generated'

## 2. Create and review training data

The generated rows are a transparent starter set from the fixed A-E catalog. They deliberately remain `human_reviewed: false`. Expand them with real deterministic scenario/evidence outputs, edit the English target summaries as a group, and mark only approved rows as reviewed before training. The splitter keeps each `case_id` in one split.

In [ ]:
from src.llm.training_data import (
    generate_seed_records, project_tool_definitions, validate_records, write_jsonl_splits,
)

seed_records = generate_seed_records()
paths = write_jsonl_splits(DATASET_DIR, seed_records)
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))
print(json.dumps(project_tool_definitions(), indent=2))

# To train: replace this with group-reviewed records from JSONL, then mark each row true.
# Example only after a real review: record['metadata']['human_reviewed'] = True
try:
    validate_records(seed_records)
except ValueError as error:
    print('Expected pre-training stop:', error)


## 3. Select reviewed records

Set this to a reviewed JSONL export rather than the starter set for a submitted experiment. The explicit validation is intentional: it prevents accidentally training on unreviewed templates.

In [ ]:
REVIEWED_JSONL = PROJECT_ROOT / 'data' / 'llm' / 'reviewed_records.jsonl'

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

if REVIEWED_JSONL.exists():
    reviewed_records = read_jsonl(REVIEWED_JSONL)
    validate_records(reviewed_records, require_human_review=True)
    print(f'Using {len(reviewed_records)} reviewed records.')
else:
    reviewed_records = []
    print(f'No reviewed dataset at {REVIEWED_JSONL}. Create it with python -m src.llm.prepare_reviewed_records (see notebooks/README.md).')


## 5. Guarded tool routing and held-out evaluation

A model response is not executed directly. The router accepts only a single approved JSON tool request, rejects extra arguments such as `allow_download`, and pins every request to the original cutoff. A full analysis should still be run through `HarnessSession`, which enforces the complete state machine.

In [ ]:
from src.llm.tool_loop import SafeProjectToolRouter, ToolRequestError

router = SafeProjectToolRouter()
safe_request = json.dumps({
    'tool_name': 'check_data_quality',
    'arguments': {'ticker': 'ASELS', 'as_of_date': '2025-09-30'},
})
print('Approved tools:', router.allowed_tools)
print('Example model request:', safe_request)
# Execution requires the repository's local market cache:
# tool_result = router.execute_json(safe_request, as_of_date='2025-09-30')

def report_guardrail_metrics(texts):
    lowered = [text.lower() for text in texts]
    return {
        'response_count': len(texts),
        'direct_trade_language': sum(('buy' in text or 'sell' in text) for text in lowered),
        'guaranteed_return_language': sum(('guaranteed' in text or 'certain return' in text) for text in lowered),
        'evidence_reference_missing': sum(('e1' not in text and 'e2' not in text) for text in lowered),
    }

# For the final report, generate base-model and adapter answers for the same held-out
# report records, then compare these metrics alongside src.experiments A-E results.


## Submission checklist

Record the base model ID, adapter directory/hash, dependency versions, seed, dataset manifest, GPU type, training parameters, and held-out guardrail results. Include representative English summaries with their evidence IDs. Do not commit the dataset, checkpoint, or adapter unless your course policy explicitly permits it.